# Phase 1: Unified OMR Dataset Loaders & Graph Representation Pipeline

This notebook provides a complete, unified data engineering pipeline for Optical Music Recognition (OMR):

### Pipeline Highlights:
1. **Camera-PrIMuS Loader**: Pairs realistic camera-distorted score slice images (`.png`) with target `.semantic` token sequences. Visualizes 5 samples to sanity check alignment.
2. **DeepScores v2 COCO Loader**: Parses COCO-format JSON bounding box annotations (`[x, y, w, h]`, categories) and visualizes score pages with class labels overlaid.
3. **MuNG XML Graph Parser**: Parses MuNG (Music Notation Graph) XML node topologies (`outlinks`/`inlinks`) and reconstructs musical note events (pitch + duration) — a dry run of Phase 3 logic.
4. **Unified Dynamic Batch Loader**: Pads dynamic score image width (`[B, 1, 64, Max_W]`) and sequence length (`[B, Max_L]`) with attention masks for model training.

In [ ]:
# Setup Environment & Path Configuration
import os
import sys
import torch
import matplotlib.pyplot as plt

# Ensure root workspace directory is in sys.path
root_dir = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from data.camera_primus_loader import CameraPrIMuSLoader
from data.deepscores_v2_loader import DeepScoresV2COCODataset
from data.mung_parser import MuNGGraphParser
from data.unified_loader import OMRDataDownloader, UnifiedOMRDataset, get_omr_dataloader

print("[SUCCESS] Environment setup complete. All OMR data pipeline modules loaded.")

## Section 1: Camera-PrIMuS Loader (Distorted Image <-> Semantic Encoding)

In [ ]:
# Load Camera-PrIMuS dataset & render 5 sanity check samples
cam_loader = CameraPrIMuSLoader(data_dir="data/camera_primus")

fig, axes = plt.subplots(5, 1, figsize=(14, 10))
print("=== CAMERA-PRIMUS SANITY CHECK (5 SAMPLES) ===")
for i in range(5):
    img, tokens, tokens_str = cam_loader.get_sample(i)
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Sample #{i+1} (Size: {img.size}, {len(tokens)} primitives)\n{tokens_str}", fontsize=9, fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## Section 2: DeepScores v2 COCO-Format Loader & Bounding Box Visualizer

In [ ]:
# Instantiate DeepScores v2 COCO dataset & visualize pages with overlaid bounding boxes
ds_coco = DeepScoresV2COCODataset(dataset_dir="data/deepscores_v2")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
print("=== DEEPSCORES V2 COCO BOUNDING BOX VISUALIZATION ===")
for page_idx in range(2):
    img, anns, img_path = ds_coco[page_idx]
    vis_img = ds_coco.visualize_boxes(img, anns)
    axes[page_idx].imshow(vis_img)
    axes[page_idx].set_title(f"DeepScores v2 Page #{page_idx+1} ({len(anns)} bounding boxes, {len(ds_coco.cat_dict)} categories)", fontsize=10, fontweight="bold")
    axes[page_idx].axis("off")

plt.tight_layout()
plt.show()

## Section 3: MuNG XML Graph Parser & Note Event Reconstruction (Pitch + Duration)

In [ ]:
# Parse MuNG XML graph topology and reconstruct musical note events
mung_parser = MuNGGraphParser(mung_xml_dir="data/mung_samples")
sample_xml = mung_parser.generate_sample_mung_xml()
note_events = mung_parser.parse_mung_xml_to_note_events(sample_xml)

print("[MUNG] === RECONSTRUCTED NOTE EVENTS (MUNG GRAPH DRY RUN) ===")
for idx, event in enumerate(note_events):
    print(f"Note Event #{idx+1} (Node ID {event.id}):")
    print(f"   - Onset Beat:     {event.onset_beat:.1f} quarter(s)")
    print(f"   - Reconstructed:  Pitch={event.pitch_name} | Duration={event.duration_name} ({event.quarter_length}q)")
    print(f"   - Clef Context:   {event.clef_context}")


## Section 4: Unified Multi-Dataset Batch Collation & DataLoader

In [ ]:
# Build Unified DataLoader with dynamic image padding & sequence batch collation
dataloader, unified_dataset = get_omr_dataloader(
    data_dir="data/primus",
    annotation_type="agnostic",
    batch_size=4,
    shuffle=True,
    img_height=64
)

batch = next(iter(dataloader))

print("=== UNIFIED BATCH TENSOR SUMMARY ===")
print("Images Tensor Shape [B, C, H, W]:", batch["images"].shape)
print("Sequences Tensor Shape [B, L]:   ", batch["sequences"].shape)
print("Padding Masks Tensor Shape [B, L]:", batch["padding_masks"].shape)
print("Batch Image Widths:              ", batch["img_widths"])
print("Batch Sequence Lengths:           ", batch["seq_lengths"])